# 消息的调用
## 1. 通过JSON或者消息对象初始化消息

举例1：

In [13]:
import asyncio
import time
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model_deepseek = init_chat_model(
    model='deepseek-flash',
    model_provider='deepseek',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

#消息初始化 JSON格式
# message = [
#     {"role": "system", "content": "你一个善于给出通俗易懂解释的AI助手"},
#     {"role": "user", "content": "你好"},
#     {"role": "assistant", "content": "你好！我能帮你什么？"},
#     {"role": "user", "content": "什么是机器学习"}
# ]
#用消息对象初始化
message = [
    SystemMessage(content="你一个善于给出通俗易懂解释的AI助手"),
    HumanMessage(content="你好"),
    AIMessage(content="你好！我能帮你什么？"),
    HumanMessage(content="什么是机器学习")

]

print("==== 程序开始 ====")
start_time = time.perf_counter()

#创建一个任务
async_task = asyncio.create_task(model_deepseek.ainvoke(message))

for i in range(3):
    await asyncio.sleep(1)
    print(f">>> 正在执行第{i + 1}个任务。。。（已经耗时{time.perf_counter() - start_time:.2f}s）")

print(">>> 本地任务执行完成")

 # 返回结果
response = await async_task

print(f" === 模型返回：{response}")
end_time = time.perf_counter()

print(f"最终时间：{end_time - start_time:.2f}s")



==== 程序开始 ====
>>> 正在执行第1个任务。。。（已经耗时0.99s）
>>> 正在执行第2个任务。。。（已经耗时2.01s）
>>> 正在执行第3个任务。。。（已经耗时3.01s）
>>> 本地任务执行完成
 === 模型返回：content='简单说：**机器学习就是让计算机从大量数据中自己找规律，然后用这些规律做判断或预测。**\n\n可以把它想成教小孩认猫：\n\n- 传统编程：你写一堆规则——“有胡须、尖耳朵、毛茸茸的就是猫”。\n- 机器学习：你给计算机看很多张猫的图片，也看很多不是猫的图片。它自己慢慢总结出“猫大概长什么样”。以后再来一张新图片，它就能判断是不是猫。\n\n所以，机器学习的核心不是人把规则写死，而是：\n\n> **数据 + 算法 → 模型 → 预测/判断**\n\n比如：\n\n- 你给邮箱看很多“垃圾邮件”和“正常邮件”，它学会以后自动分垃圾邮件。\n- 你给系统看很多人的买房数据，它学会预测房价。\n- 你给短视频 App 看你的点击、停留、点赞，它学会给你推荐可能喜欢的视频。\n\n机器学习常见分三类：\n\n1. **监督学习**：数据里有“标准答案”。  \n   比如给很多图片，并标注“这是猫 / 这不是猫”，让它学会分类。\n\n2. **无监督学习**：数据里没有标准答案。  \n   比如把一堆用户自动分成几类人，但事先不知道该怎么分。\n\n3. **强化学习**：通过试错和奖励来学习。  \n   比如下棋 AI，走得好就加分，走得差就减分，慢慢学会怎么赢。\n\n一句话总结：\n\n> **机器学习就是让机器从经验（数据）中学习，而不是靠人一条条写规则。**\n\n它不是让机器真的像人一样“有意识”，而是用数学和统计方法，从数据里找出模式，然后用来预测。  \n如果你愿意，我也可以用一个“预测房价”的小例子，把整个过程讲一遍。' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户“什么是机器学习”。需要通俗易懂解释。用户之前说“你一个善于给出通俗易懂解释的AI助手”，所以要用简单、生活化例子。需要覆盖定义、核心思想、类型、例子、与普通程序区别。不要过于技术。可以比喻：教小孩认猫，不是写规则，而是给很多猫图片，让它自己

举例2：name标签

In [17]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model_deepseek = init_chat_model(
    model='deepseek-flash',
    model_provider='deepseek',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

msg = [
    SystemMessage("你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出格式：{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}"),
    HumanMessage(content="我认为 1+1=2",name="Bob"),
    HumanMessage(content="我认为 1+1>2",name="Tom"),
    HumanMessage(content="请列出谁说了什么，不要判断对错。",name="audience")
]

response = model_deepseek.invoke(msg)

rprint(response.content)



{"speakers":[{"name":"unknown","claim":"我认为 1+1=2"},{"name":"unknown","claim":"我认为 1+1>2"}]}

举例3:tools消息的调用

In [35]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model_deepseek = init_chat_model(
    model='deepseek-flash',
    model_provider='deepseek',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},
)

def get_weather(city: str) -> str:
    return f"{city}的天气不错哦，晴空万里,但是明天可能下雨哦~"

#绑定工具方法
model_with_tools = model_deepseek.bind_tools([get_weather])
tool_call_id = "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"

ai_message = AIMessage(
    content = [],
    tool_calls = [{
        "name": "get_weather",
        "args": {"city": "北京"},
        "id": tool_call_id
    }]
)

tool_message = ToolMessage(
    content = "今天北京天气晴朗，万里无云~ ", #  get_weather("北京")
    tool_call_id = tool_call_id
)

messages_all = [
    HumanMessage(content="北京天气如何"),
    ai_message,
    tool_message,
]

result = model_with_tools.invoke(messages_all)

print(result)



content='北京今天天气晴朗，万里无云 ☀️ 适合外出活动，记得做好防晒哦！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 320, 'total_tokens': 341, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 192}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '0698e155-92ec-4152-83fc-d650d702b6be', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a09f15-5774-7dc2-96bf-114ae923bc5c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 320, 'output_tokens': 21, 'total_tokens': 341, 'input_token_details': {'cache_read': 128}, 'output_token_details': {}}
